In [2]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

In [3]:
#specifing datapath
Data_path=Path(r"C:\\Users\\User\\Downloads\\Video Game Sales dataset\\Video_Games_Sales_as_at_22_Dec_2016.csv")
# checking if file exit in the path location 
if not Data_path.exists():
    raise FileNotFoundError("CSV not found in the given file path  please check the file path ")
#load the dataset
df=pd.read_csv(Data_path)




In [4]:
#shape of the dataset
print("shape:",df.shape)


shape: (16719, 16)


In [5]:
#prints summary of coln 
display(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16717 non-null  str    
 1   Platform         16719 non-null  str    
 2   Year_of_Release  16450 non-null  float64
 3   Genre            16717 non-null  str    
 4   Publisher        16665 non-null  str    
 5   NA_Sales         16719 non-null  float64
 6   EU_Sales         16719 non-null  float64
 7   JP_Sales         16719 non-null  float64
 8   Other_Sales      16719 non-null  float64
 9   Global_Sales     16719 non-null  float64
 10  Critic_Score     8137 non-null   float64
 11  Critic_Count     8137 non-null   float64
 12  User_Score       7590 non-null   float64
 13  User_Count       7590 non-null   float64
 14  Developer        10096 non-null  str    
 15  Rating           9950 non-null   str    
dtypes: float64(10), str(6)
memory usage: 2.9 MB


None

In [6]:
#generate descriptive statistics summary
display(df.describe())

,Year_of_Release,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count
count,16450.000000,16719.000000,16719.000000,16719.000000,16719.000000,16719.000000,8137.000000,8137.000000,7590.000000,7590.000000
mean,2006.487356,0.263330,0.145025,0.077602,0.047332,0.533543,68.967679,26.360821,7.125046,162.229908
std,5.878995,0.813514,0.503283,0.308818,0.186710,1.547935,13.938165,18.980495,1.500006,561.282326
min,1980.000000,0.000000,0.000000,0.000000,0.000000,0.010000,13.000000,3.000000,0.000000,4.000000
25%,2003.000000,0.000000,0.000000,0.000000,0.000000,0.060000,60.000000,12.000000,6.400000,10.000000
50%,2007.000000,0.080000,0.020000,0.000000,0.010000,0.170000,71.000000,21.000000,7.500000,24.000000
75%,2010.000000,0.240000,0.110000,0.040000,0.030000,0.470000,79.000000,36.000000,8.200000,81.000000
max,2020.000000,41.360000,28.960000,10.220000,10.570000,82.530000,98.000000,113.000000,9.700000,10665.000000


In [7]:
df.head()

,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8.0,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8.0,192.0,Nintendo,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN


## Checking out for missing values and duplicates

In [8]:
missing_details=pd.DataFrame({
'missing_count':df.isnull().sum(),
'missing_percentage':(df.isnull().mean()*100).round(2)
}
).sort_values('missing_percentage',ascending=True)
display(missing_details[missing_details['missing_count']>0])

,missing_count,missing_percentage
Name,2,0.01
Genre,2,0.01
Publisher,54,0.32
Year_of_Release,269,1.61
Developer,6623,39.61
Rating,6769,40.49
Critic_Count,8582,51.33
Critic_Score,8582,51.33
User_Count,9129,54.60
User_Score,9129,54.60


In [9]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


Since no duplicate rows, so no need to find and remove duplicate rows

## 3. Data cleaning and validation
Use this order:
1. Inspect the raw data: check columns, types, missing values, and duplicates.
2. Remove exact duplicate rows.
3. Fix data types: convert year, scores, counts, and sales to numeric. Values like "tbd" in User_Score will become missing.
4. Clean invalid values: check for impossible score ranges, negative sales, suspicious release years, and inconsistent category text. Correct values when you can; otherwise mark them missing or document why you remove a row.
5. Remove rows missing Global_Sales: it’s the value you want to predict, so rows without it can’t train the model.
6. Handle rows missing Name: remove them if you need game names for your analysis. Name is not a predictor in the current notebook, so this choice is about data quality and reporting.
7. Keep missing predictor values for now. Don’t delete those rows automatically.
8. Split into training and test sets.
9. Fill missing predictors using training data only: numeric values with training-set medians, and missing categories with "Unknown". Apply the same fill values to the test set.
10. Check the cleaned data: confirm duplicates and invalid values are handled, the target has no missing values, and predictors are ready for modeling.
So, cleaning starts with inspection, duplicates, types, and invalid values. Filling predictor gaps is a later preprocessing step, after the split.

In [10]:
numeric_coln=df.select_dtypes('number').columns.tolist()
print(numeric_coln)

['Year_of_Release', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales', 'Critic_Score', 'Critic_Count', 'User_Score', 'User_Count']


In [11]:
for i in numeric_coln:
    if i in df.columns:
        df[i]=pd.to_numeric(df[i],errors='coerce') #errors='coerce' means the data that cant be converted will be missing value

In [12]:
df.select_dtypes('number').columns

Index(['Year_of_Release', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales',
       'Global_Sales', 'Critic_Score', 'Critic_Count', 'User_Score',
       'User_Count'],
      dtype='str')

In [13]:
df[['User_Score', 'Critic_Score']].isna().sum()

User_Score      9129
Critic_Score    8582
dtype: int64

In [14]:
# user_score range is 0-10 anything above or less is invalid 
#replacing them with nan
df.loc[
    ~df.User_Score.between(0,10),'User_Score'
]=np.nan

## Critic_score range is 0-100 anything above or less is invalid
df.loc[
    ~df.Critic_Score.between(0,100),'Critic_Score'
]=np.nan

In [15]:
df[['User_Score', 'Critic_Score']].isna().sum()

User_Score      9129
Critic_Score    8582
dtype: int64

In [16]:
df[['Critic_Count','User_Count']].isna().sum()

Critic_Count    8582
User_Count      9129
dtype: int64

In [17]:
#count cant be negative so checking and replacing it as nan if its negative
for i in ['Critic_Count','User_Count']:
    if i in df.columns:
        df.loc[df[i]<0,i]=np.nan

In [18]:
sales_coln=['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
for col in sales_coln:
    if col in df.columns:
        df.loc[df[col]<0,col]=np.nan

In [19]:
df.Year_of_Release

0        2006.0
1        1985.0
2        2008.0
3        2009.0
4        1996.0
          ...  
16714    2016.0
16715    2006.0
16716    2016.0
16717    2003.0
16718    2016.0
Name: Year_of_Release, Length: 16719, dtype: float64

In [20]:
# we remove rows having missing global sales value as its our target
if 'Global_Sales' not in df.columns:#check if column present in dataset
    raise KeyError('no Global_Sales column in the dataset')
rows_before=len(df)
df=df.dropna(subset=['Global_Sales']).copy()
print(f"Rows removed from the dataset without Global_Sales value:{rows_before-len(df)}")


Rows removed from the dataset without Global_Sales value:0


In [21]:
df.Name.shape

(16719,)

In [22]:
#dropping rows with missing Name
if 'Name' in df.columns:
    rows_before=len(df)
    df=df.dropna(subset=['Name']).copy()
    print(f"Rows removed without game name:{rows_before-len(df)}")
    

Rows removed without game name:2


In [23]:
missing_details=pd.DataFrame({
'missing_count':df.isnull().sum(),
'missing_percentage':(df.isnull().mean()*100).round(2)
}
).sort_values('missing_percentage',ascending=True)
display(missing_details[missing_details['missing_count']>0])

,missing_count,missing_percentage
Publisher,54,0.32
Year_of_Release,269,1.61
Developer,6621,39.61
Rating,6767,40.48
Critic_Count,8580,51.32
Critic_Score,8580,51.32
User_Count,9127,54.60
User_Score,9127,54.60


In [24]:
if "Year_of_Release" in df.columns:
    print("\nRelease-year summary:")
    print(df["Year_of_Release"].describe())

    print("\nMissing release years:",
          df["Year_of_Release"].isna().sum())


Release-year summary:
count    16448.000000
mean      2006.488996
std          5.877470
min       1980.000000
25%       2003.000000
50%       2007.000000
75%       2010.000000
max       2020.000000
Name: Year_of_Release, dtype: float64

Missing release years: 269


In [26]:
# Fill missing categorical values with mode

categorical_columns = [
    "Platform",
    "Genre",
    "Publisher",
    "Rating"
]

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])
    
missing_details=pd.DataFrame({
'missing_count':df.isnull().sum(),
'missing_percentage':(df.isnull().mean()*100).round(2)
}
).sort_values('missing_percentage',ascending=True)
display(missing_details[missing_details['missing_count']>0])


,missing_count,missing_percentage
Year_of_Release,269,1.61
Developer,6621,39.61
Critic_Count,8580,51.32
Critic_Score,8580,51.32
User_Count,9127,54.60
User_Score,9127,54.60
